# TopLD linkage lookup utilities

Two tools for working with TopLD linkage data:

1. **Linkage lookup** — pass an epistasis_id (or list), get R², D', and annotation back from TopLD.
2. **Region search** — specify a genomic region (chrom, start, end), get all TopLD variant pairs in that region as epistasis_ids.

TopLD data: EUR per-chromosome LD tables (`EUR_chr{N}_no_filter_0.2_1000000_LD.csv`) and annotation files (`*_info_annotation.csv`).

In [ ]:
import os, re
from pathlib import Path
from typing import Union
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

TOPLD_DIR = Path(os.environ.get('TOPLD_DIR', '/tamir2/nicolaslynn/data/TopLD/data'))
TOPLD_R2 = 0.2
TOPLD_WINDOW = 1_000_000

print(f'TOPLD_DIR: {TOPLD_DIR} (exists: {TOPLD_DIR.exists()})')

## Core functions

In [ ]:
def _ld_path(chrom):
    return TOPLD_DIR / f'EUR_chr{chrom}_no_filter_{TOPLD_R2}_{TOPLD_WINDOW}_LD.csv'


def _annot_path(chrom):
    return TOPLD_DIR / f'EUR_chr{chrom}_no_filter_{TOPLD_R2}_{TOPLD_WINDOW}_info_annotation.csv'


def parse_epistasis_id(eid):
    """Parse epistasis_id -> dict with gene, chrom, pos1, pos2, ref1, alt1, ref2, alt2, strand."""
    m1, m2 = eid.split('|')
    p1, p2 = m1.split(':'), m2.split(':')
    return {
        'gene': p1[0], 'chrom': p1[1],
        'pos1': int(p1[2]), 'pos2': int(p2[2]),
        'ref1': p1[3], 'alt1': p1[4],
        'ref2': p2[3], 'alt2': p2[4],
        'strand': p1[5] if len(p1) > 5 else 'P',
    }


def _load_annotations_for_chrom(chrom):
    """Load TopLD annotations for one chromosome -> dict Uniq_ID -> {MAF, CADD_phred, rsID, ...}."""
    path = _annot_path(chrom)
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    lookup = {}
    for _, row in df.iterrows():
        entry = {}
        for col in ['MAF', 'CADD_phred', 'rsID', 'VEP_ensembl_Gene_Name', 'VEP_ensembl_Consequence']:
            if col in df.columns and pd.notna(row.get(col)):
                val = row[col]
                if col in ('MAF', 'CADD_phred'):
                    try:
                        val = float(val)
                    except (TypeError, ValueError):
                        val = np.nan
                entry[col] = val
        lookup[str(row['Uniq_ID'])] = entry
    return lookup


print('Core functions defined.')

---
## Tool 1: Linkage lookup by epistasis_id

Pass one or more epistasis_ids. For each, we look up the pair in the TopLD LD table
for the relevant chromosome and return R², D', and any available annotations.

Returns a DataFrame with one row per query.

In [ ]:
def lookup_linkage(epistasis_ids: Union[str, list], include_annotations: bool = True) -> pd.DataFrame:
    """Look up TopLD linkage (R², D') for one or more epistasis_ids.

    Parameters
    ----------
    epistasis_ids : str or list of str
        Epistasis IDs in format GENE:chrom:pos:ref:alt:strand|GENE:chrom:pos:ref:alt:strand
    include_annotations : bool
        If True, also return MAF, CADD, rsID, gene, consequence from TopLD annotations.

    Returns
    -------
    DataFrame with columns: epistasis_id, chrom, pos1, pos2, distance,
        R2, Dprime, [+ annotation columns if requested]
    """
    if isinstance(epistasis_ids, str):
        epistasis_ids = [epistasis_ids]

    # Group queries by chromosome
    queries = []
    for eid in epistasis_ids:
        parsed = parse_epistasis_id(eid)
        queries.append({**parsed, 'epistasis_id': eid})
    qdf = pd.DataFrame(queries)

    results = []
    for chrom, group in qdf.groupby('chrom'):
        ld_path = _ld_path(chrom)
        if not ld_path.exists():
            print(f'  chr{chrom}: LD file not found')
            for _, q in group.iterrows():
                results.append({'epistasis_id': q.epistasis_id, 'chrom': chrom,
                                'pos1': q.pos1, 'pos2': q.pos2,
                                'distance': abs(q.pos2 - q.pos1),
                                'R2': np.nan, 'Dprime': np.nan, 'found_in_topld': False})
            continue

        # Build lookup set: (pmin, pmax) for this chromosome
        need_pairs = {(min(q.pos1, q.pos2), max(q.pos1, q.pos2)): q.epistasis_id
                      for _, q in group.iterrows()}
        found = set()

        # Load annotations if requested
        annot = _load_annotations_for_chrom(chrom) if include_annotations else {}

        for chunk in tqdm(pd.read_csv(ld_path, chunksize=1_000_000),
                          desc=f'chr{chrom}', leave=False):
            chunk = chunk.astype({'SNP1': int, 'SNP2': int})
            chunk['pmin'] = chunk[['SNP1', 'SNP2']].min(axis=1)
            chunk['pmax'] = chunk[['SNP1', 'SNP2']].max(axis=1)

            for _, row in chunk.iterrows():
                key = (int(row['pmin']), int(row['pmax']))
                if key in need_pairs:
                    eid = need_pairs[key]
                    entry = {
                        'epistasis_id': eid, 'chrom': chrom,
                        'pos1': key[0], 'pos2': key[1],
                        'distance': key[1] - key[0],
                        'R2': float(row['R2']), 'Dprime': float(row['Dprime']),
                        'found_in_topld': True,
                    }
                    if include_annotations:
                        for uid_col in ['Uniq_ID_1', 'Uniq_ID_2']:
                            uid = str(row.get(uid_col, ''))
                            a = annot.get(uid, {})
                            suffix = uid_col[-1]
                            for k, v in a.items():
                                entry[f'{k}_{suffix}'] = v
                    results.append(entry)
                    found.add(key)

            if len(found) == len(need_pairs):
                break

        # Mark unfound pairs
        for key, eid in need_pairs.items():
            if key not in found:
                results.append({'epistasis_id': eid, 'chrom': chrom,
                                'pos1': key[0], 'pos2': key[1],
                                'distance': key[1] - key[0],
                                'R2': np.nan, 'Dprime': np.nan, 'found_in_topld': False})

    return pd.DataFrame(results)


print('lookup_linkage() defined.')

### Example: look up linkage for specific pairs

In [ ]:
# Example usage — replace with your epistasis_ids
example_ids = [
    # 'GENE:1:924197:G:A:P|GENE:1:925548:C:T:P',
]

if example_ids:
    result = lookup_linkage(example_ids)
    display(result)
else:
    print('Add epistasis_ids above to run the lookup.')

---
## Tool 2: Search TopLD pairs in a genomic region

Specify a chromosome and position range. Returns all TopLD LD pairs within that region,
formatted as epistasis_ids with full annotations.

In [ ]:
def search_region(chrom: str, start: int, end: int,
                  min_r2: float = 0.0,
                  max_distance: int = None,
                  gene_name: str = 'GENE',
                  include_annotations: bool = True) -> pd.DataFrame:
    """Find all TopLD variant pairs within a genomic region.

    Parameters
    ----------
    chrom : str
        Chromosome (e.g., '1', '12', 'X').
    start, end : int
        Genomic coordinates (hg38, 1-based) defining the region.
    min_r2 : float
        Minimum R² to include (default 0.0 = all pairs).
    max_distance : int or None
        Maximum distance between variants in bp. None = no filter.
    gene_name : str
        Gene name to use in epistasis_id construction.
    include_annotations : bool
        If True, annotate each variant with MAF, CADD, rsID, etc.

    Returns
    -------
    DataFrame with: epistasis_id, pos1, pos2, distance, R2, Dprime,
        Uniq_ID_1, Uniq_ID_2, [+ annotation columns]
    """
    chrom = str(chrom).replace('chr', '')
    ld_path = _ld_path(chrom)
    if not ld_path.exists():
        print(f'TopLD LD file not found: {ld_path}')
        return pd.DataFrame()

    annot = _load_annotations_for_chrom(chrom) if include_annotations else {}
    matches = []

    for chunk in tqdm(pd.read_csv(ld_path, chunksize=1_000_000),
                      desc=f'Scanning chr{chrom} [{start:,}-{end:,}]', leave=False):
        chunk = chunk.astype({'SNP1': int, 'SNP2': int})

        # Both variants must be within the region
        mask = (
            (chunk['SNP1'] >= start) & (chunk['SNP1'] <= end) &
            (chunk['SNP2'] >= start) & (chunk['SNP2'] <= end)
        )
        if min_r2 > 0:
            mask &= chunk['R2'] >= min_r2

        m = chunk[mask].copy()
        if m.empty:
            # If we've passed the region, stop early
            if chunk['SNP1'].min() > end:
                break
            continue

        m['distance'] = (m['SNP2'] - m['SNP1']).abs()
        if max_distance is not None:
            m = m[m.distance <= max_distance]

        if not m.empty:
            matches.append(m)

    if not matches:
        print(f'No TopLD pairs found in chr{chrom}:{start}-{end}')
        return pd.DataFrame()

    df = pd.concat(matches, ignore_index=True)

    # Build epistasis_ids
    def _make_eid(row):
        uid1, uid2 = str(row['Uniq_ID_1']), str(row['Uniq_ID_2'])
        # Uniq_ID format: pos:ref:alt
        p1 = uid1.split(':')
        p2 = uid2.split(':')
        return (f"{gene_name}:{chrom}:{p1[0]}:{p1[1]}:{p1[2]}:P"
                f"|{gene_name}:{chrom}:{p2[0]}:{p2[1]}:{p2[2]}:P")

    df['epistasis_id'] = df.apply(_make_eid, axis=1)

    # Annotate
    if include_annotations and annot:
        for suffix, uid_col in [('1', 'Uniq_ID_1'), ('2', 'Uniq_ID_2')]:
            for col_name in ['MAF', 'CADD_phred', 'rsID', 'VEP_ensembl_Consequence']:
                df[f'{col_name}_{suffix}'] = df[uid_col].apply(
                    lambda uid: annot.get(str(uid), {}).get(col_name, np.nan)
                )

    out_cols = ['epistasis_id', 'SNP1', 'SNP2', 'distance', 'R2', 'Dprime',
                'Uniq_ID_1', 'Uniq_ID_2']
    annot_cols = [c for c in df.columns if c.startswith(('MAF_', 'CADD_', 'rsID_', 'VEP_'))]
    out_cols.extend(sorted(annot_cols))
    out_cols = [c for c in out_cols if c in df.columns]

    print(f'Found {len(df)} pairs in chr{chrom}:{start:,}-{end:,}')
    return df[out_cols].reset_index(drop=True)


print('search_region() defined.')

### Example: search KRAS region (chr12)

In [ ]:
# Example: KRAS neighborhood on chr12
# kras_pairs = search_region('12', 25_227_193, 25_227_494, gene_name='KRAS')
# display(kras_pairs.head(10))

print('Uncomment the lines above and set your region to search.')

---
## Batch linkage lookup from a pairs file

Look up linkage for all pairs in `all_pairs_combined.tsv` (or any subset).

In [ ]:
def batch_linkage_lookup(pairs_tsv: Path, source: str = None, max_pairs: int = None) -> pd.DataFrame:
    """Look up TopLD linkage for pairs from a TSV file.

    Parameters
    ----------
    pairs_tsv : Path
        Path to TSV with 'epistasis_id' and optionally 'source' columns.
    source : str or None
        Filter to a specific source (e.g., 'yang_evqtl'). None = all.
    max_pairs : int or None
        Limit number of pairs to look up. None = all.

    Returns
    -------
    DataFrame with linkage results.
    """
    df = pd.read_csv(pairs_tsv, sep='\t', usecols=['epistasis_id', 'source'], low_memory=False)
    if source:
        df = df[df.source == source]
    eids = df['epistasis_id'].drop_duplicates().tolist()
    if max_pairs:
        eids = eids[:max_pairs]
    print(f'Looking up {len(eids)} pairs...')
    return lookup_linkage(eids, include_annotations=True)


# Example:
# import sys; sys.path.insert(0, str(Path('..').resolve()))
# from paper_data_config import EPISTASIS_PAPER_ROOT
# result = batch_linkage_lookup(EPISTASIS_PAPER_ROOT / 'data' / 'all_pairs_combined.tsv',
#                                source='yang_evqtl', max_pairs=10)
# display(result)
print('batch_linkage_lookup() defined. Uncomment example above to run.')